In [1]:
from pathlib import Path
import os


# --- Helper: choose between macOS-style and Windows-style roots ---
def project_path(mac_path_str, win_path_str):
    mac_p = Path(mac_path_str)
    win_p = Path(win_path_str)
    if mac_p.exists():
        return mac_p
    elif win_p.exists():
        return win_p
    else:
        # Neither exists; return mac version but warn
        print(f"WARNING: Neither {mac_p} nor {win_p} exists on this system.")
        return mac_p


project_path_set = project_path(
    "/Users/kaankeskin/projects/FrequencySliding/",
    "C:/Users/kaank/OneDrive/Belgeler/GitHub/FrequencySliding/"
)

# Set the working directory
os.chdir(project_path_set)

# Check current working directory
print("Current working directory:", os.getcwd())

Current working directory: /Users/kaankeskin/projects/FrequencySliding


In [ ]:
import pandas as pd
pre_df=pd.read_csv("./data/output/tau_perROI.csv",index_col=0)
post_df=pd.read_csv("./data/output/tau_perROI_AfterECT.csv",index_col=0)


# ── 2. Clean up the subject IDs so they match ──────────────────────────
def clean_pre(name):
    # "sub-1.results"  →  "sub-1"
    return name.replace(".results", "")

def clean_post(name):
    # "sub-1_ses2.results" → "sub-1"
    return name.replace("_ses2.results", "")

pre_df = pre_df.rename(index=clean_pre)
post_df = post_df.rename(index=clean_post)

# ── 3. Restrict to the intersection of subjects ───────────────────────
common_subjects = pre_df.index.intersection(post_df.index)
print(common_subjects)
pre_df  = pre_df.loc[common_subjects]
post_df = post_df.loc[common_subjects]

# 3. Melt each to long form
df_pre_long = (
    pre_df
    .reset_index()                                   # bring 'subject' back as a column
    .melt(id_vars="subject", var_name="region", value_name="value")
    .assign(session=0)               # 0 = pre
)
df_post_long = (
    post_df
    .reset_index()
    .melt(id_vars="subject", var_name="region", value_name="value")
    .assign(session=1)               # 1 = post
)

# 4. Make sure region is integer
df_pre_long ["region"] = df_pre_long["region"].astype(int)
df_post_long["region"] = df_post_long["region"].astype(int)

# 5. Inner-merge on subject & region
df_long = pd.concat([df_pre_long, df_post_long], ignore_index=True)

# 6. Drop rows where value is NaN
df_long = df_long.dropna(subset=["value"])

# 7. (Optionally) make categorical
df_long["session"] = df_long["session"].astype("category")
df_long["subject"] = df_long["subject"].astype("category")

print(df_long)



Index(['sub-1', 'sub-13', 'sub-19', 'sub-20', 'sub-23', 'sub-4', 'sub-5',
       'sub-6', 'sub-8', 'sub-33', 'sub-36', 'sub-38', 'sub-39'],
      dtype='object', name='subject')
     subject  region     value session
0      sub-1       0  0.146558       0
1     sub-13       0  0.454147       0
2     sub-19       0  0.436107       0
3     sub-20       0  0.712064       0
4     sub-23       0  0.453828       0
...      ...     ...       ...     ...
9348  sub-13     359  0.315881       1
9353   sub-5     359  0.085998       1
9354   sub-6     359  0.046385       1
9356  sub-33     359  0.011089       1
9357  sub-36     359  0.840655       1

[6144 rows x 4 columns]


In [26]:
# after you’ve built df_long with columns subject, region, value, session:

# Keep only those subject–region pairs that have both session 0 and 1
df_long = df_long[
    # For each row, computes how many distinct sessions that row’s (subject,region) has.
    df_long.groupby(["subject", "region"])["session"]
           .transform("nunique")
           .eq(2) #Returns True only if that pair has exactly 2 sessions (i.e. pre and post).
].reset_index(drop=True)

import pandas as pd
import numpy as np
from pathlib import Path

# --- Paths ---
ctrl_path = Path(r"./data/raw")
ctrl_csv = ctrl_path / "controlregions.csv"

# --- Read CSV ---
# We'll not assume headers; we'll read and rename first two cols defensively.
ctrl_df = pd.read_csv(ctrl_csv)

df_long["unimodal"] = df_long["region"].map(ctrl_df["unimodal"])
# if you’d rather call it “region_category” with labels:
df_long["region_category"] = df_long["unimodal"].map({
    1: "unimodal",   # or True: "unimodal"
    0: "transmodal"  # or False: "transmodal"
}).astype("category")
print(df_long)

df_long.to_csv("./data/output/tau_BeforeAfterECT.csv",index=True)


     subject  region     value session  unimodal region_category
0      sub-1       0  0.146558       0         1        unimodal
1      sub-1       0  0.918447       1         1        unimodal
2      sub-1       3  0.093133       0         1        unimodal
3      sub-1       3  0.396346       1         1        unimodal
4      sub-1       4  0.463886       0         1        unimodal
...      ...     ...       ...     ...       ...             ...
4199   sub-8     349  0.085155       1         0      transmodal
4200   sub-8     350  0.092094       0         0      transmodal
4201   sub-8     350  0.080840       1         0      transmodal
4202   sub-8     352  0.078926       0         1        unimodal
4203   sub-8     352  0.089115       1         1        unimodal

[4204 rows x 6 columns]


/var/folders/mx/j9rl9nns0g7bz9pd9t16prh00000gn/T/ipykernel_71947/1093564386.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_long.groupby(["subject", "region"])["session"]
